In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# A Theorem Prover for First-Order Logic without Equality

## Auxiliary Functions

The resolution calculus works with clauses.  The notebook `09-FOL-CNF.ipynb` implements the function `normalize` that turns a formula into a set of clauses.  It also defines the types `Literal` and `Clause`:
```
type Literal =
    | Pos of string * Term list   // p(t1, ..., tn)
    | Neg of string * Term list   // ¬p(t1, ..., tn)

type Clause = Set<Literal>
```
This notebook imports the notebook `FOL-Parser.ipynb`, which provides the function `parse` that converts a string into a formula.  The parser distinguishes variables and function symbols as follows:
- A word starting with a lower case letter is interpreted as a *variable*.
- A word starting with an upper case letter is assumed to be a *function* or *predicate symbol*.

In [2]:
importNotebook "09-FOL-CNF.ipynb"

imported 

FOL-Parser.ipynb, 09-FOL-CNF.ipynb

In [3]:
let s = "∀g:∀c:(Grandparent(g, c) ↔ ∃p: (Parent(g, p) ∧ Parent(p, c)))"
let f = parse s
formulaToString f

"∀g:∀c:(Grandparent(g, c) ↔ ∃p:(Parent(g, p) ∧ Parent(p, c)))"

In [4]:
let Clauses = normalize f
printfn "%s" (clausesToString Clauses)

{
    {Grandparent(g, c), ¬Parent(a, c), ¬Parent(g, a)},
    {Parent(g, sk2(g, c)), ¬Grandparent(g, c)},
    {Parent(sk2(g, c), c), ¬Grandparent(g, c)}
}

The notebook `10-Unification.ipynb` implements [unification](https://en.wikipedia.org/wiki/Unification_(computer_science)) via the algorithm of [Martelli and Montanari](https://dl.acm.org/doi/pdf/10.1145/357162.357169).  It provides the function `unify`.

In [5]:
importNotebook "10-Unification.ipynb"

imported 

10-Unification.ipynb

The function `arb` takes one argument:
- `S` is a non-empty set.

It returns an arbitrary element from the set $S$, namely the smallest element.

In [6]:
let arb (S: Set<'T>) : 'T =
    Set.minElement S

The function `complement` takes one argument:
- `l` is a literal.

It returns the complement $\overline{\,l\,}$ of the literal $l$.

In [7]:
let complement (l: Literal) : Literal =
    match l with
    | Pos (p, ts) -> Neg (p, ts)
    | Neg (p, ts) -> Pos (p, ts)

In [8]:
literalToString (complement (Pos ("P", [ Var "x" ])))

"¬P(x)"

In [9]:
literalToString (complement (Neg ("P", [ Var "x" ])))

"P(x)"

The function `literalVariables` takes one argument:
- `l` is a literal.

It returns the set of all variables occurring in `l`.  The function `collectVariables` takes a clause `C` and returns the set of all variables occurring in `C`.  The function `termVariables` is defined in the notebook `09-FOL-CNF.ipynb`.

In [10]:
let literalVariables (l: Literal) : Set<string> =
    match l with
    | Pos (_, ts) | Neg (_, ts) -> Set.unionMany (List.map termVariables ts)

let collectVariables (C: Clause) : Set<string> =
    Set.unionMany (Seq.map literalVariables C)

In [11]:
for C in Clauses do
    printfn "collectVariables %s = \n\t%A" (clauseToString C) (collectVariables C)

collectVariables 

{Grandparent(g, c), ¬Parent(a, c), ¬Parent(g, a)}

 = 
	

set ["a"; "c"; "g"]

collectVariables 

...

set ["c"; "g"]

collectVariables 

{Parent(sk2(g, c), c), ¬Grandparent(g, c)}

 = 
	

set ["c"; "g"]

The function `applyLiteral` takes two arguments:
- `σ` is a substitution,
- `l` is a literal.

It applies `σ` to all terms occurring in `l`.  The function `applyClause` applies a substitution `σ` to all literals of a clause `C`.  The function `applyTerm` is defined in the notebook `10-Unification.ipynb`.

In [12]:
let applyLiteral (σ: Map<string, Term>) (l: Literal) : Literal =
    match l with
    | Pos (p, ts) -> Pos (p, List.map (applyTerm σ) ts)
    | Neg (p, ts) -> Neg (p, List.map (applyTerm σ) ts)

let applyClause (σ: Map<string, Term>) (C: Clause) : Clause =
    Set.map (applyLiteral σ) C

The function `renameVariables` takes two arguments:
- `f` is a clause,
- `g` is a clause.

It renames the variables in the clause `f` so that they are different from the variables occurring in `g`.  The new variables are single lower case letters.

In [13]:
let renameVariables (f: Clause) (g: Clause) : Clause =
    let OldVars = Set.toList (collectVariables f)
    let NewVars = [ for c in 'a' .. 'z' do
                      if not ((collectVariables g).Contains (string c)) then Var (string c) ]
    let σ = Map (List.zip OldVars (List.take OldVars.Length NewVars))
    applyClause σ f

In [14]:
for C in Clauses do
    printfn "%s" (clauseToString (renameVariables C C))

{Grandparent(e, d), ¬Parent(b, d), ¬Parent(e, b)}

{Parent(b, sk2(b, a)), ¬Grandparent(b, a)}

{Parent(sk2(b, a), a), ¬Grandparent(b, a)}

In order to unify two literals, we convert them into terms.  The function `literalToTerm` takes one argument:
- `l` is a literal.

It returns a term representing `l`:  The literal $p(t_1, \cdots, t_n)$ is represented as the term $p(t_1, \cdots, t_n)$, while $\neg p(t_1, \cdots, t_n)$ is represented as the term $\neg\bigl(p(t_1, \cdots, t_n)\bigr)$, where $\neg$ is treated as a unary function symbol.  Then two literals are unifiable if and only if the corresponding terms are unifiable.

The function `unifyLiterals` takes two literals and returns their most general unifier, or `None` if they are not unifiable.

In [15]:
let literalToTerm (l: Literal) : Term =
    match l with
    | Pos (p, ts) -> Fun (p, ts)
    | Neg (p, ts) -> Fun ("¬", [ Fun (p, ts) ])

let unifyLiterals (l1: Literal) (l2: Literal) : Map<string, Term> option =
    unify (literalToTerm l1) (literalToTerm l2)

# A Calculus for First Order Logic

The [resolution](https://en.wikipedia.org/wiki/Resolution_(logic)) rule is an inference rule that is defined as follows: If
 * $C_1$ and $C_2$ are clauses from first order logic,
 * $p(s_1,\cdots,s_n)$ and $p(t_1,\cdots,t_n)$ are atomic formulas,
 * the syntactical equation $p(s_1,\cdots,s_n) \doteq p(t_1,\cdots,t_n)$ is solvable and
     $$ \mu = \mathtt{mgu}\bigl(p(s_1,\cdots,s_n), p(t_1,\cdots,t_n)\bigr), $$
then
$$\frac{C_1 \cup\{ p(s_1,\cdots,s_n)\} \quad\quad \{\neg p(t_1,\cdots,t_n)\} \cup C_2}{
                 C_1\mu \cup C_2\mu}
$$
is an application of the resolution rule.

The function `resolve` takes two arguments:
- `C1` is a clause,
- `C2` is a clause.

It returns the set of all clauses that can be inferred from `C1` and `C2` by applying the resolution rule.  Before the rule is applied, the variables of `C2` are renamed, so that `C1` and `C2` have no variables in common.

In [16]:
let resolve (C1: Clause) (C2: Clause) : Set<Clause> =
    let C2 = renameVariables C2 C1
    set [ for L1 in C1 do
            for L2 in C2 do
                match unifyLiterals L1 (complement L2) with
                | Some μ -> applyClause μ ((C1.Remove L1) + (C2.Remove L2))
                | None   -> () ]

## Some Formulas for Testing

According to <a href="https://de.wikipedia.org/wiki/Uwe_Schöning">Uwe Schöning</a>, the theory of red dragons is
given by the following axioms:
<ol>
<li>
Every dragon is happy if all its children can fly:
$$ \forall x: \Bigl(\forall y: \big(\texttt{Child}(y,x) \rightarrow \texttt{CanFly}(y)\big) \rightarrow \texttt{Happy}(x)\Bigr)
$$
</li>
<li>
All red dragons can fly:
$$
 \forall x: \bigl(\texttt{Red}(x) \rightarrow \texttt{CanFly}(x)\bigr)
$$
</li>
<li> The children of red dragons are themselves red:
$$
\forall x: \bigl(\texttt{Red}(x) \rightarrow \forall y:\bigl( \texttt{Child}(y,x) \rightarrow \texttt{Red}(y)\bigr)\bigr)
$$
</li>
</ol>
We will show that these axioms imply that all red dragons are happy:
$$
 \forall x: \bigl(\texttt{Red}(x) \rightarrow \texttt{Happy}(x)\bigr)
$$
To this end, the formula stating that all red dragons are happy is negated.  Then we will show that the set consisting of the negated formula together with the axioms is inconsistent.  We start by defining the formulas.

In [17]:
let s1 = "∀x:(∀y:(Child(y, x) → CanFly(y)) → Happy(x))"
let s2 = "∀x:(Red(x) → CanFly(x))"
let s3 = "∀x:(Red(x) → ∀y:(Child(y, x) → Red(y)))"
let s4 = "¬∀x:(Red(x) → Happy(x))"

Next, the formulas are parsed and transformed into clauses.

In [18]:
let f1 = parse s1
printfn "%s" (clausesToString (normalize f1))

{
    {Child(sk3(x), x), Happy(x)},
    {Happy(x), ¬CanFly(sk3(x))}
}

In [19]:
let f2 = parse s2
printfn "%s" (clausesToString (normalize f2))

{
    {CanFly(x), ¬Red(x)}
}

In [20]:
let f3 = parse s3
printfn "%s" (clausesToString (normalize f3))

{
    {Red(y), ¬Child(y, x), ¬Red(x)}
}

In [21]:
let f4 = parse s4
printfn "%s" (clausesToString (normalize f4))

{
    {Red(sk4())},
    {¬Happy(sk4())}
}

In [22]:
let DragonClauses = normalize f1 + normalize f2 + normalize f3 + normalize f4
printfn "%s" (clausesToString DragonClauses)

{
    {CanFly(x), ¬Red(x)},
    {Child(sk5(x), x), Happy(x)},
    {Happy(x), ¬CanFly(sk5(x))},
    {Red(y), ¬Child(y, x), ¬Red(x)},
    {Red(sk6())},
    {¬Happy(sk6())}
}

We give names to the clauses in order to be able to refer to them.  The names of the Skolem functions computed by `normalize` depend on how often a Skolem function has been created before.  In the clauses below, we use the names `sk2` for the Skolem function of the first axiom and `sk3` for the Skolem constant of the negated formula.  Of course, the names of the Skolem functions are irrelevant for the proof.

In [23]:
let C1 : Clause = set [ Pos ("Red", [ Fun ("sk3", []) ]) ]
let C2 : Clause = set [ Neg ("Happy", [ Fun ("sk3", []) ]) ]
let C3 : Clause = set [ Pos ("CanFly", [ Var "x" ]); Neg ("Red", [ Var "x" ]) ]
let C4 : Clause = set [ Pos ("Child", [ Fun ("sk2", [ Var "x" ]); Var "x" ]); Pos ("Happy", [ Var "x" ]) ]
let C5 : Clause = set [ Pos ("Happy", [ Var "x" ]); Neg ("CanFly", [ Fun ("sk2", [ Var "x" ]) ]) ]
let C6 : Clause = set [ Pos ("Red", [ Var "y" ]); Neg ("Child", [ Var "y"; Var "x" ]); Neg ("Red", [ Var "x" ]) ]

Now we are ready to show that the set consisting of these clauses is inconsistent.

In [24]:
let C7 = arb (resolve C1 C6)
clauseToString C7

"{Red(b), ¬Child(b, sk3())}"

In [25]:
let C8 = arb (resolve C7 C4)
clauseToString C8

"{Happy(sk3()), Red(sk2(sk3()))}"

In [26]:
let C9 = arb (resolve C8 C2)
clauseToString C9

"{Red(sk2(sk3()))}"

In [27]:
let C10 = arb (resolve C9 C3)
clauseToString C10

"{CanFly(sk2(sk3()))}"

In [28]:
let C11 = arb (resolve C10 C5)
clauseToString C11

"{Happy(sk3())}"

In [29]:
clauseToString (arb (resolve C11 C2))

"{}"

As we have derived the empty clause, we have shown that all <b style="color:red;">communist dragons</b> are happy!

## Factorization

A calculus which only contains the resolution rule is not complete.  We also need the <em style="color:blue;">factorization</em> rule.  If
<ol>
<li> $C$ is a clause from first order logic,</li>
<li> $p(s_1,\cdots,s_n)$ and $p(t_1,\cdots,t_n)$ are atomic formulas,</li>
<li> the syntactical equation $p(s_1,\cdots,s_n)  \doteq p(t_1,\cdots,t_n)$ is solvable and
     $$\mu = \mathtt{mgu}\bigl(p(s_1,\cdots,s_n), p(t_1,\cdots,t_n)\bigr), $$</li>
</ol>
then both

$\displaystyle \frac{C \cup \bigl\{p(s_1,\cdots,s_n),\, p(t_1,\cdots,t_n)\bigl\}}{C\mu \cup \bigl\{p(s_1,\cdots,s_n)\mu\bigr\} } $
and
$\displaystyle \frac{C \cup \bigl\{ \neg p(s_1,\cdots,s_n),\, \neg p(t_1,\cdots,t_n)\bigl\}}{C\mu \cup \bigl\{\neg p(s_1,\cdots,s_n)\mu\bigr\} }
$

are applications of the factorization rule.

The function `factorize` takes one argument:
- `C` is a clause from first order logic.

It returns the set of all clauses that can be derived from $C$ via factorization.

In [30]:
let factorize (C: Clause) : Set<Clause> =
    set [ for L1 in C do
            for L2 in C do
                if L1 <> L2 then
                    match unifyLiterals L1 L2 with
                    | Some μ -> applyClause μ C
                    | None   -> () ]

The clauses
$$C_1 := \forall x: \forall y: P(F(x),y) \vee \forall u: \forall v:P(u, G(v))$$
and
$$C_2 := \forall x: \forall y: \bigl(\neg P(F(x),y)\bigr) \vee \forall u: \forall v: \bigl(\neg P(u, G(v))\bigr)$$
are inconsistent.  However, the resolution rule alone is not sufficient to show this.

In [31]:
let D1 = arb (normalize (parse "∀x:∀y:P(F(x),y) ∨ ∀u:∀v:P(u,G(v))"))
clauseToString D1

"{P(u, G(v)), P(F(x), y)}"

In [32]:
let D2 = arb (normalize (parse "∀x:∀y:(¬P(F(x),y)) ∨ ∀u:∀v:(¬P(u,G(v)))"))
clauseToString D2

"{¬P(u, G(v)), ¬P(F(x), y)}"

In [33]:
let D3 = arb (factorize D1)
clauseToString D3

"{P(F(x), G(v))}"

In [34]:
let D4 = arb (factorize D2)
clauseToString D4

"{¬P(F(x), G(v))}"

In [35]:
clauseToString (arb (resolve D3 D4))

"{}"

## Automatic Theorem Proving

In order to be able to construct a proof once the empty clause has been derived, we have to remember how every clause has been derived.  A clause is derived either by resolution from two clauses or by factorization from one clause.  This is expressed by the type `Reason`.

In [36]:
type Reason =
    | Resolution    of Clause * Clause
    | Factorization of Clause

The function `infere` takes one argument:
- `Clauses` is a set of clauses.

It returns the set of all pairs `(C, reason)` such that the clause `C` results from
<ol>
    <li>the resolution of two clauses $C_1, C_2 \in \texttt{Clauses}$, or</li>
    <li>the factorization of a clause $C_1 \in \texttt{Clauses}$.</li>
</ol>
The second component `reason` records how `C` has been derived.

In [37]:
let infere (Clauses: Set<Clause>) : Set<Clause * Reason> =
    set [ for C1 in Clauses do
            for C2 in Clauses do
                for C in resolve C1 C2 -> (C, Resolution (C1, C2)) ]
    + set [ for C1 in Clauses do
              for C in factorize C1 -> (C, Factorization C1) ]

The function `saturate` takes one argument:
- `Cs` is a set of clauses.

It tries to infer the empty clause from `Cs`.  It returns a map that assigns to every derived clause the reason how this clause has been derived.  If it is not possible to infer the empty clause, the function runs until memory is exhausted.

**Implementation:** In every iteration, all clauses that can be inferred from the current set of clauses are computed.  The clauses that are new are added to the set `Clauses` and their reasons are stored in the map `Reasons`.

In [38]:
let saturate (Cs: Set<Clause>) : Map<Clause, Reason> =
    let mutable Clauses = Cs
    let mutable Reasons : Map<Clause, Reason> = Map.empty
    let mutable cnt = 1
    while not (Clauses.Contains Set.empty) do
        for (C, R) in infere Clauses do
            if not (Clauses.Contains C) then
                Reasons <- Reasons.Add(C, R)
                Clauses <- Clauses.Add C
        printfn "cnt = %d, number of clauses: %d" cnt Clauses.Count
        cnt <- cnt + 1
    Reasons

In [39]:
printfn "%s" (clausesToString DragonClauses)

{
    {CanFly(x), ¬Red(x)},
    {Child(sk5(x), x), Happy(x)},
    {Happy(x), ¬CanFly(sk5(x))},
    {Red(y), ¬Child(y, x), ¬Red(x)},
    {Red(sk6())},
    {¬Happy(sk6())}
}

In [40]:
#!time
let Reasons = saturate DragonClauses

cnt = 

1

, number of clauses: 

18

cnt = 

...

cnt = 

3

, number of clauses: 

6963

Wall time: 1652.9024ms

The function `constructProof` takes two arguments:
- `clause` is a clause that has been derived,
- `Reasons` is a map that records how the clauses have been derived.

It returns a proof of `clause` as a list of strings, where every string describes one step of the proof.  Clauses that do not have a reason are axioms.  The auxiliary function `update` takes two proofs `P1` and `P2` and appends those lines of `P2` to `P1` that do not already occur in `P1`.

In [41]:
let update (P1: string list) (P2: string list) : string list =
    P1 @ [ for line in P2 do if not (List.contains line P1) then line ]

let rec constructProof (clause: Clause) (Reasons: Map<Clause, Reason>) : string list =
    match Reasons.TryFind clause with
    | None ->
        [ sprintf "Axiom:       %s" (clauseToString clause) ]
    | Some (Factorization c) ->
        constructProof c Reasons
        @ [ sprintf "Factorization: %s \n⊢%s%s" (clauseToString c) (String.replicate 12 " ") (clauseToString clause) ]
    | Some (Resolution (c1, c2)) ->
        let proof1 = constructProof c1 Reasons
        let proof2 = constructProof c2 Reasons
        update proof1 proof2
        @ [ sprintf "Resolution:  %s,\n%s%s  \n⊢%s%s" (clauseToString c1) (String.replicate 13 " ")
                    (clauseToString c2) (String.replicate 12 " ") (clauseToString clause) ]

In [42]:
let Proof = constructProof Set.empty Reasons
for line in Proof do
    printfn "%s" line

Axiom:       {Child(sk5(x), x), Happy(x)}

Axiom:       {¬Happy(sk6())}

Resolution:  {Child(sk5(x), x), Happy(x)},
             {¬Happy(sk6())}  
⊢            {Child(sk5(sk6()), sk6())}

Axiom:       {Red(sk6())}

Axiom:       {Red(y), ¬Child(y, x), ¬Red(x)}

...

Axiom:       {CanFly(x), ¬Red(x)}

Axiom:       {Happy(x), ¬CanFly(sk5(x))}

Resolution:  {Happy(x), ¬CanFly(sk5(x))},
             {¬Happy(sk6())}  
⊢            {¬CanFly(sk5(sk6()))}

Resolution:  {CanFly(x), ¬Red(x)},
             {¬CanFly(sk5(sk6()))}  
⊢            {¬Red(sk5(sk6()))}

Resolution:  {Red(sk5(sk6()))},
             {¬Red(sk5(sk6()))}  
⊢            {}